In [1]:
import pandas as pd
import spacy
nlp = spacy.load("en_core_web_sm",disable=["parser", "ner"])
df = pd.read_csv(r"C:\Users\Lenovo\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\A6A521C45BF54CB40C63CBB6383398495351B936\transfers\2026-10\hotel_reviews.csv")
df.head()
df = df[["Rating(Out of 10)","Review_Text"]]
df.dropna(inplace=True)
df.head()

,Rating(Out of 10),Review_Text
0,9.0,Hotel the pearl is perfect place to stay in De...
1,9.0,Location of the hotel is perfect. The hotel is...
2,9.0,"Location, Indian food."
3,9.0,The location and the hotel itself is great. Ne...
4,9.0,Friendly and smiling staffs.. The reception st...


In [2]:
def creat(t):
    if t>7:
        return "Positive"
    else:
        return "Nagative"
df["sentiment"] = df["Rating(Out of 10)"].apply(creat)
df = df.drop(columns="Rating(Out of 10)",axis=1)
df.dropna(inplace=True)
df.head()

,Review_Text,sentiment
0,Hotel the pearl is perfect place to stay in De...,Positive
1,Location of the hotel is perfect. The hotel is...,Positive
2,"Location, Indian food.",Positive
3,The location and the hotel itself is great. Ne...,Positive
4,Friendly and smiling staffs.. The reception st...,Positive


In [3]:
print(len(df))

6994


In [4]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

clean_texts = []

for doc in nlp.pipe(df["Review_Text"].astype(str), batch_size=100):
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
    clean_texts.append(" ".join(tokens))

df["clean_text"] = clean_texts
df = df.drop(columns="Review_Text",axis=1)
df.head()

,sentiment,clean_text
0,Positive,hotel pearl perfect place stay delhi paharganj...
1,Positive,location hotel perfect hotel peaceful nice sta...
2,Positive,location indian food
3,Positive,location hotel great time stay nice room comfo...
4,Positive,friendly smile staff reception staff excellent...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
vectorizer = TfidfVectorizer(max_features=5000)
X  = vectorizer.fit_transform(df["clean_text"])
y = df["sentiment"]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [6]:
from sklearn.metrics import accuracy_score,r2_score
model  = LogisticRegression()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("test",accuracy_score(y_test,y_pred))
print("train",accuracy_score(y_train,(model.predict(X_train))))

test 0.781987133666905
train 0.8259159964253798


In [7]:
def predict(inpt):
    doc = list(nlp.pipe([inpt]))[0]
    
    tokens = [token.lemma_.lower() for token in doc 
              if not token.is_stop and not token.is_punct]
    
    clean_text = " ".join(tokens)
    
    review = vectorizer.transform([clean_text])
    prediction = model.predict(review)
    
    return prediction[0]

user = input("enter")
result = predict(user)
result

enter this hotel was very amazing


'Positive'

In [9]:
import joblib

joblib.dump(model, "sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [10]:
import streamlit as st
import joblib
import spacy

# load model
model = joblib.load("sentiment_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# predict function
def predict(inpt):
    doc = list(nlp.pipe([inpt]))[0]
    
    tokens = [token.lemma_.lower() for token in doc 
              if not token.is_stop and not token.is_punct]
    
    clean_text = " ".join(tokens)
    
    review = vectorizer.transform([clean_text])
    prediction = model.predict(review)
    
    return prediction[0]

# UI
st.title("Hotel Review Sentiment Analysis")

user_input = st.text_area("Enter your review:")

if st.button("Predict"):
    if user_input:
        result = predict(user_input)
        st.success(f"Sentiment: {result}")
    else:
        st.warning("Please enter text")

2026-04-24 09:45:44.878 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45:45.071 
  command:

    streamlit run C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-24 09:45:45.073 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45:45.074 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45:45.076 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45:45.077 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45:45.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-24 09:45

In [ ]:

def clean(text):
    doc = nlp(str(text))   # safe conversion
    tokens = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            tokens.append(token.lemma_.lower())

    return " ".join(tokens)
df["clean_text"] = df["Review_Text"].apply(clean)
df.head()